In [10]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.tree import DecisionTreeClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE


In [2]:
TH_500 = pd.read_csv(
    "../classification/TomsHardware/Relative_labeling/sigma=500/TomsHardware-Relative-Sigma-500.data",
    sep=",",
    header=None
)

TH_1000= pd.read_csv(
    "../classification/TomsHardware/Relative_labeling/sigma=1000/TomsHardware-Relative-Sigma-1000.data",
    sep=",",
    header=None
)
TH_1500= pd.read_csv(
    "../classification/TomsHardware/Relative_labeling/sigma=1500/TomsHardware-Relative-Sigma-1500.data",
    sep=",",
    header=None
)
groups = [
    "NCD", "BL", "NAD", "AI", "NAC", "ND",
    "CS", "AT", "NA", "ADL", "AS_NA", "AS_NAC"
]

columns = []
for group in groups:
    for t in range(8):
        columns.append(f"{group}_{t}")

columns.append("label")  

TH_500.columns = columns
TH_1000.columns = columns
TH_1500.columns = columns

prefixes = {col.split("_")[0] for col in TH_500.columns if "_" in col}
prefixes = {col.split("_")[0] for col in TH_1000.columns if "_" in col}
prefixes = {col.split("_")[0] for col in TH_1500.columns if "_" in col}

### 500

### 1 Baseline Decision Tree

In [5]:
X = TH_500.drop("label", axis=1)
y = TH_500["label"]

X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.2,random_state=42, stratify=y)

tree_baseline = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)

y_pred_tree = tree_baseline.predict(X_test)
y_prob_tree = tree_baseline.predict_proba(X_test)[:,1]


accuracy_tree = accuracy_score(y_test, y_pred_tree)
precision_tree = precision_score(y_test, y_pred_tree)
recall_tree = recall_score(y_test, y_pred_tree)
f1_tree = f1_score(y_test, y_pred_tree)
roc_auc_tree = roc_auc_score(y_test, y_prob_tree)

cm_tree = confusion_matrix(y_test, y_pred_tree)

print('1️.Baseline Decision Tree')
print("Accuracy:", accuracy_tree)
print("Precision:", precision_tree)
print("Recall:", recall_tree)
print("F1-score:", f1_tree)
print("ROC-AUC:", roc_auc_tree)

print("\nConfusion Matrix:")
print(cm_tree)

1️.Baseline Decision Tree
Accuracy: 0.8918406072106262
Precision: 0.7535816618911175
Recall: 0.7557471264367817
F1-score: 0.7546628407460545
ROC-AUC: 0.8429992728696479

Confusion Matrix:
[[1147   86]
 [  85  263]]


### ️2.Balanced Decision Tree

In [6]:
tree_balanced = DecisionTreeClassifier(random_state=42,class_weight="balanced").fit(X_train, y_train)

y_pred_tree_bal = tree_balanced.predict(X_test)
y_prob_tree_bal = tree_balanced.predict_proba(X_test)[:,1]

accuracy_tree_bal = accuracy_score(y_test, y_pred_tree_bal)
precision_tree_bal = precision_score(y_test, y_pred_tree_bal)
recall_tree_bal = recall_score(y_test, y_pred_tree_bal)
f1_tree_bal = f1_score(y_test, y_pred_tree_bal)
roc_auc_tree_bal = roc_auc_score(y_test, y_prob_tree_bal)

cm_tree_bal = confusion_matrix(y_test, y_pred_tree_bal)

print('2.Balanced Decision Tree')
print("Accuracy:", accuracy_tree_bal)
print("Precision:", precision_tree_bal)
print("Recall:", recall_tree_bal)
print("F1-score:", f1_tree_bal)
print("ROC-AUC:", roc_auc_tree_bal)

print("\nConfusion Matrix:")
print(cm_tree_bal)

2.Balanced Decision Tree
Accuracy: 0.8905755850727388
Precision: 0.7725856697819314
Recall: 0.7126436781609196
F1-score: 0.7414050822122571
ROC-AUC: 0.8267192437844338

Confusion Matrix:
[[1160   73]
 [ 100  248]]


### 3.Decision Tree with Stratified K-Fold Cross-Validation

In [7]:
tree_model = DecisionTreeClassifier(random_state=42)

cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results_tree = cross_validate( tree_model,X,y,cv=cv,scoring=scoring)

print('3.Decision Tree with Stratified K-Fold Cross-Validation')
print("Accuracy:", cv_results_tree["test_accuracy"].mean())
print("Precision:", cv_results_tree["test_precision"].mean())
print("Recall:", cv_results_tree["test_recall"].mean())
print("F1-score:", cv_results_tree["test_f1"].mean())
print("ROC-AUC:", cv_results_tree["test_roc_auc"].mean())

3.Decision Tree with Stratified K-Fold Cross-Validation
Accuracy: 0.8867805186590765
Precision: 0.7401740566101898
Recall: 0.7495636136086685
F1-score: 0.7447081706983234
ROC-AUC: 0.8375492755480802


### 4.Decision Tree with Grid Search

In [9]:
tree = DecisionTreeClassifier(random_state=42)

param_grid = {
    "max_depth": [3, 5, 10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "criterion": ["gini", "entropy"]
}

grid_tree = GridSearchCV(tree, param_grid,cv=5,scoring="f1", n_jobs=-1).fit(X_train, y_train)
best_tree = grid_tree.best_estimator_


y_pred_tree_grid = best_tree.predict(X_test)
y_prob_tree_grid = best_tree.predict_proba(X_test)[:,1]

accuracy_tree_grid = accuracy_score(y_test, y_pred_tree_grid)
precision_tree_grid = precision_score(y_test, y_pred_tree_grid)
recall_tree_grid = recall_score(y_test, y_pred_tree_grid)
f1_tree_grid = f1_score(y_test, y_pred_tree_grid)
roc_auc_tree_grid = roc_auc_score(y_test, y_prob_tree_grid)

cm_tree_grid = confusion_matrix(y_test, y_pred_tree_grid)

print('4.Decision Tree with Grid Search')
print("Best parameters:", grid_tree.best_params_)
print("Accuracy:", accuracy_tree_grid)
print("Precision:", precision_tree_grid)
print("Recall:", recall_tree_grid)
print("F1-score:", f1_tree_grid)
print("ROC-AUC:", roc_auc_tree_grid)

print("\nConfusion Matrix:")
print(cm_tree_grid)

4.Decision Tree with Grid Search
Best parameters: {'criterion': 'gini', 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}
Accuracy: 0.9101834282099937
Precision: 0.7395348837209302
Recall: 0.9137931034482759
F1-score: 0.8174807197943444
ROC-AUC: 0.9588565409104044

Confusion Matrix:
[[1121  112]
 [  30  318]]


### 5.Decision Tree with Grid Search with SMOTE

In [11]:
pipeline = Pipeline([("smote", SMOTE(random_state=42)),("tree", DecisionTreeClassifier(random_state=42))])


param_grid = {
    "tree__max_depth": [3, 5, 10, 20, None],
    "tree__min_samples_split": [2, 5, 10],
    "tree__min_samples_leaf": [1, 2, 5],
    "tree__criterion": ["gini", "entropy"]
}

grid_tree_smote = GridSearchCV(pipeline,param_grid,cv=5, scoring="f1",n_jobs=-1).fit(X_train, y_train)


best_tree_smote = grid_tree_smote.best_estimator_
y_pred_tree_smote = best_tree_smote.predict(X_test)
y_prob_tree_smote = best_tree_smote.predict_proba(X_test)[:,1]

accuracy_tree_smote = accuracy_score(y_test, y_pred_tree_smote)
precision_tree_smote = precision_score(y_test, y_pred_tree_smote)
recall_tree_smote = recall_score(y_test, y_pred_tree_smote)
f1_tree_smote = f1_score(y_test, y_pred_tree_smote)
roc_auc_tree_smote = roc_auc_score(y_test, y_prob_tree_smote)

cm_tree_smote = confusion_matrix(y_test, y_pred_tree_smote)

print('Decision Tree with Grid Search with SMOTE')
print("Best parameters:", grid_tree_smote.best_params_)
print("Accuracy:", accuracy_tree_smote)
print("Precision:", precision_tree_smote)
print("Recall:", recall_tree_smote)
print("F1-score:", f1_tree_smote)
print("ROC-AUC:", roc_auc_tree_smote)

print("\nConfusion Matrix:")
print(cm_tree_smote)

Decision Tree with Grid Search with SMOTE
Best parameters: {'tree__criterion': 'entropy', 'tree__max_depth': 3, 'tree__min_samples_leaf': 1, 'tree__min_samples_split': 2}
Accuracy: 0.8905755850727388
Precision: 0.6782077393075356
Recall: 0.9568965517241379
F1-score: 0.7938021454112039
ROC-AUC: 0.9600777470145706

Confusion Matrix:
[[1075  158]
 [  15  333]]


Для датасету σ = 500 було проведено п’ять експериментів із використанням Decision Tree. Базова модель показала Accuracy = 0.892, F1-score = 0.755 та ROC-AUC = 0.843. Використання class_weight = balanced не призвело до покращення результатів, оскільки F1-score зменшився до 0.741, а Recall до 0.713.

Оцінювання за допомогою Stratified K-Fold Cross-Validation показало стабільні результати (F1-score = 0.745, ROC-AUC = 0.838), що підтверджує надійність моделі.

Найкращі результати було отримано після оптимізації гіперпараметрів методом Grid Search, де модель із параметром max_depth = 3 досягла Accuracy = 0.910, F1-score = 0.817 та ROC-AUC = 0.959. Це свідчить про те, що обмеження глибини дерева значно зменшує overfitting і покращує якість класифікації.

Використання SMOTE дозволило ще більше підвищити Recall до 0.957, що означає майже повне виявлення позитивного класу. Однак це супроводжувалося зниженням Precision до 0.678, що призвело до більшої кількості помилкових позитивних прогнозів.

In [1]:
import pandas as pd

data = [
    {
        "model": "Decision Tree (baseline)",
        "accuracy": 0.8918406072106262,
        "precision": 0.7535816618911175,
        "recall": 0.7557471264367817,
        "f1_score": 0.7546628407460545,
        "roc_auc": 0.8429992728696479
    },
    {
        "model": "Decision Tree (balanced)",
        "accuracy": 0.8905755850727388,
        "precision": 0.7725856697819314,
        "recall": 0.7126436781609196,
        "f1_score": 0.7414050822122571,
        "roc_auc": 0.8267192437844338
    },
    {
        "model": "Decision Tree (Stratified K-Fold)",
        "accuracy": 0.8867805186590765,
        "precision": 0.7401740566101898,
        "recall": 0.749563636086685,
        "f1_score": 0.7447081706983234,
        "roc_auc": 0.8375492755480802
    },
    {
        "model": "Decision Tree (Grid Search)",
        "accuracy": 0.9101834282099937,
        "precision": 0.7395348837209302,
        "recall": 0.9137931034482759,
        "f1_score": 0.8174807197943444,
        "roc_auc": 0.9588565409104044
    },
    {
        "model": "Decision Tree (Grid Search with SMOTE)",
        "accuracy": 0.8905755850727388,
        "precision": 0.6782077393075356,
        "recall": 0.9568965517241379,
        "f1_score": 0.7938021454112039,
        "roc_auc": 0.9600777470145706
    }
]

df = pd.DataFrame(data)

df

,model,accuracy,precision,recall,f1_score,roc_auc
0,Decision Tree (baseline),0.891841,0.753582,0.755747,0.754663,0.842999
1,Decision Tree (balanced),0.890576,0.772586,0.712644,0.741405,0.826719
2,Decision Tree (Stratified K-Fold),0.886781,0.740174,0.749564,0.744708,0.837549
3,Decision Tree (Grid Search),0.910183,0.739535,0.913793,0.817481,0.958857
4,Decision Tree (Grid Search with SMOTE),0.890576,0.678208,0.956897,0.793802,0.960078


### 1000

### 1 Baseline Decision Tree

In [18]:
X = TH_1000.drop("label", axis=1)
y = TH_1000["label"]

X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.2,random_state=42, stratify=y)

tree_baseline = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)

y_pred_tree = tree_baseline.predict(X_test)
y_prob_tree = tree_baseline.predict_proba(X_test)[:,1]


accuracy_tree = accuracy_score(y_test, y_pred_tree)
precision_tree = precision_score(y_test, y_pred_tree)
recall_tree = recall_score(y_test, y_pred_tree)
f1_tree = f1_score(y_test, y_pred_tree)
roc_auc_tree = roc_auc_score(y_test, y_prob_tree)

cm_tree = confusion_matrix(y_test, y_pred_tree)

print('1️.Baseline Decision Tree')
print("Accuracy:", accuracy_tree)
print("Precision:", precision_tree)
print("Recall:", recall_tree)
print("F1-score:", f1_tree)
print("ROC-AUC:", roc_auc_tree)

print("\nConfusion Matrix:")
print(cm_tree)

1️.Baseline Decision Tree
Accuracy: 0.9285262492093611
Precision: 0.7738693467336684
Recall: 0.6936936936936937
F1-score: 0.7315914489311164
ROC-AUC: 0.8302905554561185

Confusion Matrix:
[[1314   45]
 [  68  154]]


### 2.Balanced Decision Tree 

In [19]:
tree_balanced = DecisionTreeClassifier(random_state=42,class_weight="balanced").fit(X_train, y_train)

y_pred_tree_bal = tree_balanced.predict(X_test)
y_prob_tree_bal = tree_balanced.predict_proba(X_test)[:,1]

accuracy_tree_bal = accuracy_score(y_test, y_pred_tree_bal)
precision_tree_bal = precision_score(y_test, y_pred_tree_bal)
recall_tree_bal = recall_score(y_test, y_pred_tree_bal)
f1_tree_bal = f1_score(y_test, y_pred_tree_bal)
roc_auc_tree_bal = roc_auc_score(y_test, y_prob_tree_bal)

cm_tree_bal = confusion_matrix(y_test, y_pred_tree_bal)

print('2.Balanced Decision Tree')
print("Accuracy:", accuracy_tree_bal)
print("Precision:", precision_tree_bal)
print("Recall:", recall_tree_bal)
print("F1-score:", f1_tree_bal)
print("ROC-AUC:", roc_auc_tree_bal)

print("\nConfusion Matrix:")
print(cm_tree_bal)

2.Balanced Decision Tree
Accuracy: 0.9259962049335864
Precision: 0.7560975609756098
Recall: 0.6981981981981982
F1-score: 0.7259953161592506
ROC-AUC: 0.8307032197760674

Confusion Matrix:
[[1309   50]
 [  67  155]]


### 3.Decision Tree with Stratified K-Fold Cross-Validation

In [20]:
tree_model = DecisionTreeClassifier(random_state=42)

cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results_tree = cross_validate( tree_model,X,y,cv=cv,scoring=scoring)

print('3.Decision Tree with Stratified K-Fold Cross-Validation')
print("Accuracy:", cv_results_tree["test_accuracy"].mean())
print("Precision:", cv_results_tree["test_precision"].mean())
print("Recall:", cv_results_tree["test_recall"].mean())
print("F1-score:", cv_results_tree["test_f1"].mean())
print("ROC-AUC:", cv_results_tree["test_roc_auc"].mean())

3.Decision Tree with Stratified K-Fold Cross-Validation
Accuracy: 0.9287792536369386
Precision: 0.7426681072714312
Recall: 0.7558558558558559
F1-score: 0.7488844622254863
ROC-AUC: 0.8564415408786269


### 4.Decision Tree with Grid Search

In [21]:
tree = DecisionTreeClassifier(random_state=42)

param_grid = {
    "max_depth": [3, 5, 10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "criterion": ["gini", "entropy"]
}

grid_tree = GridSearchCV(tree, param_grid,cv=5,scoring="f1", n_jobs=-1).fit(X_train, y_train)
best_tree = grid_tree.best_estimator_


y_pred_tree_grid = best_tree.predict(X_test)
y_prob_tree_grid = best_tree.predict_proba(X_test)[:,1]

accuracy_tree_grid = accuracy_score(y_test, y_pred_tree_grid)
precision_tree_grid = precision_score(y_test, y_pred_tree_grid)
recall_tree_grid = recall_score(y_test, y_pred_tree_grid)
f1_tree_grid = f1_score(y_test, y_pred_tree_grid)
roc_auc_tree_grid = roc_auc_score(y_test, y_prob_tree_grid)

cm_tree_grid = confusion_matrix(y_test, y_pred_tree_grid)

print('4.Decision Tree with Grid Search')
print("Best parameters:", grid_tree.best_params_)
print("Accuracy:", accuracy_tree_grid)
print("Precision:", precision_tree_grid)
print("Recall:", recall_tree_grid)
print("F1-score:", f1_tree_grid)
print("ROC-AUC:", roc_auc_tree_grid)

print("\nConfusion Matrix:")
print(cm_tree_grid)

4.Decision Tree with Grid Search
Best parameters: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 2}
Accuracy: 0.9430740037950665
Precision: 0.7946428571428571
Recall: 0.8018018018018018
F1-score: 0.7982062780269058
ROC-AUC: 0.9613355076931236

Confusion Matrix:
[[1313   46]
 [  44  178]]


### 5.Decision Tree with Grid Search with SMOTE

In [22]:
pipeline = Pipeline([("smote", SMOTE(random_state=42)),("tree", DecisionTreeClassifier(random_state=42))])


param_grid = {
    "tree__max_depth": [3, 5, 10, 20, None],
    "tree__min_samples_split": [2, 5, 10],
    "tree__min_samples_leaf": [1, 2, 5],
    "tree__criterion": ["gini", "entropy"]
}

grid_tree_smote = GridSearchCV(pipeline,param_grid,cv=5, scoring="f1",n_jobs=-1).fit(X_train, y_train)


best_tree_smote = grid_tree_smote.best_estimator_
y_pred_tree_smote = best_tree_smote.predict(X_test)
y_prob_tree_smote = best_tree_smote.predict_proba(X_test)[:,1]

accuracy_tree_smote = accuracy_score(y_test, y_pred_tree_smote)
precision_tree_smote = precision_score(y_test, y_pred_tree_smote)
recall_tree_smote = recall_score(y_test, y_pred_tree_smote)
f1_tree_smote = f1_score(y_test, y_pred_tree_smote)
roc_auc_tree_smote = roc_auc_score(y_test, y_prob_tree_smote)

cm_tree_smote = confusion_matrix(y_test, y_pred_tree_smote)

print('Decision Tree with Grid Search with SMOTE')
print("Best parameters:", grid_tree_smote.best_params_)
print("Accuracy:", accuracy_tree_smote)
print("Precision:", precision_tree_smote)
print("Recall:", recall_tree_smote)
print("F1-score:", f1_tree_smote)
print("ROC-AUC:", roc_auc_tree_smote)

print("\nConfusion Matrix:")
print(cm_tree_smote)

Decision Tree with Grid Search with SMOTE
Best parameters: {'tree__criterion': 'entropy', 'tree__max_depth': 5, 'tree__min_samples_leaf': 5, 'tree__min_samples_split': 2}
Accuracy: 0.9165085388994307
Precision: 0.6388888888888888
Recall: 0.9324324324324325
F1-score: 0.7582417582417582
ROC-AUC: 0.9669719388262434

Confusion Matrix:
[[1242  117]
 [  15  207]]


Для датасету σ = 1000 було проведено п’ять експериментів із використанням Decision Tree. Базова модель показала Accuracy = 0.929, F1-score = 0.732 та ROC-AUC = 0.830. Використання class_weight = balanced не призвело до суттєвого покращення результатів, оскільки F1-score залишився приблизно на тому ж рівні (0.726).

Оцінювання за допомогою Stratified K-Fold Cross-Validation показало стабільні результати (F1-score = 0.749, ROC-AUC = 0.856), що підтверджує надійність моделі.

Найкращі результати було отримано після оптимізації гіперпараметрів методом Grid Search, де модель із параметрами max_depth = 5 та min_samples_leaf = 2 досягла Accuracy = 0.943, F1-score = 0.798 та ROC-AUC = 0.961.

Застосування SMOTE дозволило значно підвищити Recall до 0.932, що означає значне покращення виявлення позитивного класу. Однак це призвело до зниження Precision до 0.639 та Accuracy до 0.917, що свідчить про збільшення кількості помилкових позитивних прогнозів.

In [2]:
import pandas as pd

data = [
    {
        "model": "Decision Tree (baseline)",
        "accuracy": 0.9285262492093611,
        "precision": 0.7738693467336684,
        "recall": 0.6936936936936937,
        "f1_score": 0.7315914489311164,
        "roc_auc": 0.8302905554561185
    },
    {
        "model": "Decision Tree (balanced)",
        "accuracy": 0.9259962049335864,
        "precision": 0.7560975609756098,
        "recall": 0.6981981981981982,
        "f1_score": 0.7259953161592506,
        "roc_auc": 0.8307032197760674
    },
    {
        "model": "Decision Tree (Stratified K-Fold)",
        "accuracy": 0.9287792536369386,
        "precision": 0.7426681072714312,
        "recall": 0.7558558558558559,
        "f1_score": 0.7488844622254863,
        "roc_auc": 0.8564415408786269
    },
    {
        "model": "Decision Tree (Grid Search)",
        "accuracy": 0.9430740037950665,
        "precision": 0.7946428571428571,
        "recall": 0.8018018018018018,
        "f1_score": 0.7982062780269058,
        "roc_auc": 0.9613355076931236
    },
    {
        "model": "Decision Tree (Grid Search + SMOTE)",
        "accuracy": 0.9165085388994307,
        "precision": 0.6388888888888888,
        "recall": 0.9324324324324325,
        "f1_score": 0.7582417582417582,
        "roc_auc": 0.9669719388262434
    }
]

df = pd.DataFrame(data)
df

,model,accuracy,precision,recall,f1_score,roc_auc
0,Decision Tree (baseline),0.928526,0.773869,0.693694,0.731591,0.830291
1,Decision Tree (balanced),0.925996,0.756098,0.698198,0.725995,0.830703
2,Decision Tree (Stratified K-Fold),0.928779,0.742668,0.755856,0.748884,0.856442
3,Decision Tree (Grid Search),0.943074,0.794643,0.801802,0.798206,0.961336
4,Decision Tree (Grid Search + SMOTE),0.916509,0.638889,0.932432,0.758242,0.966972


### 1500

### 1 Baseline Decision Tree

In [23]:
X = TH_1500.drop("label", axis=1)
y = TH_1500["label"]

X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.2,random_state=42, stratify=y)

tree_baseline = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)

y_pred_tree = tree_baseline.predict(X_test)
y_prob_tree = tree_baseline.predict_proba(X_test)[:,1]


accuracy_tree = accuracy_score(y_test, y_pred_tree)
precision_tree = precision_score(y_test, y_pred_tree)
recall_tree = recall_score(y_test, y_pred_tree)
f1_tree = f1_score(y_test, y_pred_tree)
roc_auc_tree = roc_auc_score(y_test, y_prob_tree)

cm_tree = confusion_matrix(y_test, y_pred_tree)

print('1️.Baseline Decision Tree')
print("Accuracy:", accuracy_tree)
print("Precision:", precision_tree)
print("Recall:", recall_tree)
print("F1-score:", f1_tree)
print("ROC-AUC:", roc_auc_tree)

print("\nConfusion Matrix:")
print(cm_tree)

1️.Baseline Decision Tree
Accuracy: 0.9437065148640101
Precision: 0.7453416149068323
Recall: 0.7142857142857143
F1-score: 0.729483282674772
ROC-AUC: 0.8426347184308968

Confusion Matrix:
[[1372   41]
 [  48  120]]


### 2.Balanced Decision Tree

In [24]:
tree_balanced = DecisionTreeClassifier(random_state=42,class_weight="balanced").fit(X_train, y_train)

y_pred_tree_bal = tree_balanced.predict(X_test)
y_prob_tree_bal = tree_balanced.predict_proba(X_test)[:,1]

accuracy_tree_bal = accuracy_score(y_test, y_pred_tree_bal)
precision_tree_bal = precision_score(y_test, y_pred_tree_bal)
recall_tree_bal = recall_score(y_test, y_pred_tree_bal)
f1_tree_bal = f1_score(y_test, y_pred_tree_bal)
roc_auc_tree_bal = roc_auc_score(y_test, y_prob_tree_bal)

cm_tree_bal = confusion_matrix(y_test, y_pred_tree_bal)

print('2.Balanced Decision Tree')
print("Accuracy:", accuracy_tree_bal)
print("Precision:", precision_tree_bal)
print("Recall:", recall_tree_bal)
print("F1-score:", f1_tree_bal)
print("ROC-AUC:", roc_auc_tree_bal)

print("\nConfusion Matrix:")
print(cm_tree_bal)

2.Balanced Decision Tree
Accuracy: 0.9500316255534472
Precision: 0.8156028368794326
Recall: 0.6845238095238095
F1-score: 0.7443365695792881
ROC-AUC: 0.8330616216762713

Confusion Matrix:
[[1387   26]
 [  53  115]]


### 3.Decision Tree with Stratified K-Fold Cross-Validation

In [25]:
tree_model = DecisionTreeClassifier(random_state=42)

cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results_tree = cross_validate( tree_model,X,y,cv=cv,scoring=scoring)

print('3.Decision Tree with Stratified K-Fold Cross-Validation')
print("Accuracy:", cv_results_tree["test_accuracy"].mean())
print("Precision:", cv_results_tree["test_precision"].mean())
print("Recall:", cv_results_tree["test_recall"].mean())
print("F1-score:", cv_results_tree["test_f1"].mean())
print("ROC-AUC:", cv_results_tree["test_roc_auc"].mean())

3.Decision Tree with Stratified K-Fold Cross-Validation
Accuracy: 0.9466160657811511
Precision: 0.7501906088405446
Recall: 0.7479430825584672
F1-score: 0.7488366367378463
ROC-AUC: 0.8591072399413934


### 4.Decision Tree with Grid Search

In [26]:
tree = DecisionTreeClassifier(random_state=42)

param_grid = {
    "max_depth": [3, 5, 10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "criterion": ["gini", "entropy"]
}

grid_tree = GridSearchCV(tree, param_grid,cv=5,scoring="f1", n_jobs=-1).fit(X_train, y_train)
best_tree = grid_tree.best_estimator_


y_pred_tree_grid = best_tree.predict(X_test)
y_prob_tree_grid = best_tree.predict_proba(X_test)[:,1]

accuracy_tree_grid = accuracy_score(y_test, y_pred_tree_grid)
precision_tree_grid = precision_score(y_test, y_pred_tree_grid)
recall_tree_grid = recall_score(y_test, y_pred_tree_grid)
f1_tree_grid = f1_score(y_test, y_pred_tree_grid)
roc_auc_tree_grid = roc_auc_score(y_test, y_prob_tree_grid)

cm_tree_grid = confusion_matrix(y_test, y_pred_tree_grid)

print('4.Decision Tree with Grid Search')
print("Best parameters:", grid_tree.best_params_)
print("Accuracy:", accuracy_tree_grid)
print("Precision:", precision_tree_grid)
print("Recall:", recall_tree_grid)
print("F1-score:", f1_tree_grid)
print("ROC-AUC:", roc_auc_tree_grid)

print("\nConfusion Matrix:")
print(cm_tree_grid)

4.Decision Tree with Grid Search
Best parameters: {'criterion': 'gini', 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}
Accuracy: 0.9620493358633776
Precision: 0.7842105263157895
Recall: 0.8869047619047619
F1-score: 0.8324022346368715
ROC-AUC: 0.971485020051899

Confusion Matrix:
[[1372   41]
 [  19  149]]


### 5.Decision Tree with Grid Search with SMOTE

In [27]:
pipeline = Pipeline([("smote", SMOTE(random_state=42)),("tree", DecisionTreeClassifier(random_state=42))])


param_grid = {
    "tree__max_depth": [3, 5, 10, 20, None],
    "tree__min_samples_split": [2, 5, 10],
    "tree__min_samples_leaf": [1, 2, 5],
    "tree__criterion": ["gini", "entropy"]
}

grid_tree_smote = GridSearchCV(pipeline,param_grid,cv=5, scoring="f1",n_jobs=-1).fit(X_train, y_train)


best_tree_smote = grid_tree_smote.best_estimator_
y_pred_tree_smote = best_tree_smote.predict(X_test)
y_prob_tree_smote = best_tree_smote.predict_proba(X_test)[:,1]

accuracy_tree_smote = accuracy_score(y_test, y_pred_tree_smote)
precision_tree_smote = precision_score(y_test, y_pred_tree_smote)
recall_tree_smote = recall_score(y_test, y_pred_tree_smote)
f1_tree_smote = f1_score(y_test, y_pred_tree_smote)
roc_auc_tree_smote = roc_auc_score(y_test, y_prob_tree_smote)

cm_tree_smote = confusion_matrix(y_test, y_pred_tree_smote)

print('Decision Tree with Grid Search with SMOTE')
print("Best parameters:", grid_tree_smote.best_params_)
print("Accuracy:", accuracy_tree_smote)
print("Precision:", precision_tree_smote)
print("Recall:", recall_tree_smote)
print("F1-score:", f1_tree_smote)
print("ROC-AUC:", roc_auc_tree_smote)

print("\nConfusion Matrix:")
print(cm_tree_smote)

Decision Tree with Grid Search with SMOTE
Best parameters: {'tree__criterion': 'entropy', 'tree__max_depth': 5, 'tree__min_samples_leaf': 2, 'tree__min_samples_split': 2}
Accuracy: 0.9512966476913346
Precision: 0.7058823529411765
Recall: 0.9285714285714286
F1-score: 0.8020565552699229
ROC-AUC: 0.9785326728001887

Confusion Matrix:
[[1348   65]
 [  12  156]]


Для датасету σ = 1500 було проведено п’ять експериментів із використанням Decision Tree. Базова модель показала Accuracy = 0.944, F1-score = 0.729 та ROC-AUC = 0.843. Використання class_weight = balanced призвело лише до незначного покращення F1-score до 0.744.

Оцінювання за допомогою Stratified K-Fold Cross-Validation показало стабільні результати (F1-score = 0.749, ROC-AUC = 0.859).

Найкращі результати були отримані після оптимізації гіперпараметрів методом Grid Search, де модель із параметром max_depth = 3 досягла Accuracy = 0.962, F1-score = 0.832 та ROC-AUC = 0.971.

Застосування SMOTE дозволило значно підвищити Recall до 0.929, однак це супроводжувалося зниженням Precision до 0.706, що призвело до трохи нижчого F1-score = 0.802.

In [5]:
import pandas as pd

data = [
    {
        "model": "Decision Tree (baseline)",
        "accuracy": 0.9437065148640101,
        "precision": 0.7453416149068323,
        "recall": 0.7142857142857143,
        "f1_score": 0.729483286474772,
        "roc_auc": 0.8426347184308968
    },
    {
        "model": "Decision Tree (balanced)",
        "accuracy": 0.9500316255534472,
        "precision": 0.8156028368794326,
        "recall": 0.6845238095238095,
        "f1_score": 0.7443365695792881,
        "roc_auc": 0.8330616216762713
    },
    {
        "model": "Decision Tree (Stratified K-Fold)",
        "accuracy": 0.9466106057811511,
        "precision": 0.7501906088405446,
        "recall": 0.7479439252336448,
        "f1_score": 0.7488366367378463,
        "roc_auc": 0.8591072399413934
    },
    {
        "model": "Decision Tree (Grid Search)",
        "accuracy": 0.9620493358633776,
        "precision": 0.7842105263157895,
        "recall": 0.8869047619047619,
        "f1_score": 0.8324022346368715,
        "roc_auc": 0.971485020051899
    },
    {
        "model": "Decision Tree (Grid Search with SMOTE)",
        "accuracy": 0.9512966476913346,
        "precision": 0.7058823529411765,
        "recall": 0.9285714285714286,
        "f1_score": 0.8020565552699229,
        "roc_auc": 0.9785326728001887
    }
]

df = pd.DataFrame(data)

df

,model,accuracy,precision,recall,f1_score,roc_auc
0,Decision Tree (baseline),0.943707,0.745342,0.714286,0.729483,0.842635
1,Decision Tree (balanced),0.950032,0.815603,0.684524,0.744337,0.833062
2,Decision Tree (Stratified K-Fold),0.946611,0.750191,0.747944,0.748837,0.859107
3,Decision Tree (Grid Search),0.962049,0.784211,0.886905,0.832402,0.971485
4,Decision Tree (Grid Search with SMOTE),0.951297,0.705882,0.928571,0.802057,0.978533


### Result 

σ = 500

Для датасету з σ = 500 базова модель показала Accuracy = 0.892, F1-score = 0.755 та ROC-AUC = 0.843. Використання class_weight = balanced не призвело до покращення результатів, оскільки F1-score зменшився до 0.741.

Оцінювання за допомогою Stratified K-Fold Cross-Validation підтвердило стабільність моделі (F1-score = 0.745, ROC-AUC = 0.838).

Найкращі результати було отримано після Grid Search оптимізації, де модель із параметром max_depth = 3 досягла Accuracy = 0.910, F1-score = 0.817 та ROC-AUC = 0.959. Використання SMOTE дозволило підвищити Recall до 0.957, однак F1-score зменшився до 0.794 через зниження Precision.

σ = 1000

Для датасету з σ = 1000 базова модель показала Accuracy = 0.929, F1-score = 0.732 та ROC-AUC = 0.830. Balanced модель не показала суттєвого покращення (F1-score = 0.726).

Cross-validation продемонструвала стабільні результати (F1-score = 0.749, ROC-AUC = 0.856).

Найкраща продуктивність була отримана після Grid Search, де модель із параметрами max_depth = 5 та min_samples_leaf = 2 досягла Accuracy = 0.943, F1-score = 0.798 та ROC-AUC = 0.961.

Застосування SMOTE підвищило Recall до 0.932, однак зменшило Precision, що призвело до F1-score = 0.758.

σ = 1500

Для датасету з σ = 1500 базова модель показала Accuracy = 0.944, F1-score = 0.729 та ROC-AUC = 0.843. Balanced модель дала лише незначне покращення (F1-score = 0.744).

Cross-validation показала стабільні результати (F1-score = 0.749, ROC-AUC = 0.859).

Найкращі результати було отримано після Grid Search оптимізації, де модель із параметром max_depth = 3 досягла Accuracy = 0.962, F1-score = 0.832 та ROC-AUC = 0.971.

Використання SMOTE дозволило значно підвищити Recall до 0.929, однак через зниження Precision значення F1-score зменшилося до 0.802.

Результати показують, що оптимізація гіперпараметрів методом Grid Search є найефективнішим способом покращення Decision Tree для всіх трьох датасетів. Саме ця конфігурація забезпечує найвищі значення Accuracy, F1-score та ROC-AUC.

Застосування class balancing не призвело до значного покращення якості моделі. Натомість використання SMOTE значно підвищує Recall, що покращує здатність моделі знаходити позитивний клас, але одночасно знижує Precision, що призводить до меншого значення F1-score.

Також спостерігається тенденція покращення результатів зі збільшенням параметра σ. Найкращі показники були отримані для датасету σ = 1500, де модель досягла Accuracy = 0.962, F1-score = 0.832 та ROC-AUC = 0.971.